In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

### Preparação


In [6]:
from pathlib import Path
import hashlib
import json
from processamento import (
    CLASSES, ROTULOS_PASTAS, limpar_texto, tokenizar,
    transformar_textos, texto_para_tfidf, extrair_caracteristicas,
)

RAIZ = Path.cwd()
DATASET = RAIZ / "dataset_caed"
if not DATASET.is_dir():
    raise FileNotFoundError("Execute o notebook na raiz que contém dataset_caed.")
RESULTADOS = RAIZ / "resultados"
RESULTADOS.mkdir(exist_ok=True)
SEMENTE = 42

### Leitura e Auditoria

In [7]:
registros = []
for arquivo in sorted(DATASET.rglob("*.txt")):
    classe = ROTULOS_PASTAS[arquivo.parent.name]
    texto = limpar_texto(arquivo.read_text(encoding="utf-8-sig"))
    registros.append({
        "documento": arquivo.relative_to(DATASET).as_posix(),
        "texto": texto,
        "classe": classe,
        "n_palavras": len(tokenizar(texto)),
        "hash_texto": hashlib.sha256(texto.lower().encode("utf-8")).hexdigest(),
    })

df = pd.DataFrame(registros)
if df.empty:
    raise ValueError("Nenhum arquivo .txt encontrado.")
print("Total original:", len(df))
print(df.groupby("classe")["n_palavras"].agg(["count", "min", "median", "max"]))

vazios = df[df["n_palavras"] == 0].copy()
df = df[df["n_palavras"] > 0].copy()
rotulos_por_hash = df.groupby("hash_texto")["classe"].nunique()
hashes_conflitantes = rotulos_por_hash[rotulos_por_hash > 1].index
conflitos = df[df["hash_texto"].isin(hashes_conflitantes)].copy()
df = df[~df["hash_texto"].isin(hashes_conflitantes)].copy()
duplicatas = df[df.duplicated("hash_texto", keep="first")].copy()
df = df.drop_duplicates("hash_texto").reset_index(drop=True)

for nome, tabela in [("vazios", vazios), ("conflitos", conflitos),
                     ("duplicatas", duplicatas)]:
    tabela.drop(columns="texto").to_csv(RESULTADOS / f"{nome}.csv", index=False)
    print(nome, len(tabela), "arquivos")
print("Documentos restantes:", len(df))
assert set(df["classe"]) == set(CLASSES)

Total original: 2076
                       count  min  median  max
classe                                        
Ensino Fundamental I     297  295   326.0  579
Ensino Fundamental II    325  284   369.0  617
Ensino Médio             628  263   347.0  748
Ensino Superior          826  266   357.0  803
vazios 0 arquivos
conflitos 24 arquivos
duplicatas 4 arquivos
Documentos restantes: 2048
